# LinkageX — Dataset Audit for NLP Structuring

**Goal:** Audit `LINKAGEX_MASTER_CASES_V2.csv` before training semantic NLP models.

This notebook does not train models or silently modify labels. It identifies missingness, class imbalance, potential label contradictions, repeated text patterns, annotation-source differences, and linkage-ID structure.

**Important:** The dataset appears synthetic. Treat model results as validation on this benchmark, not evidence of real-world cybercrime prevalence or performance. Review flagged examples manually before excluding or correcting anything.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import hashlib
from IPython.display import display

DATA_PATH = Path("C:\\MAJOR PROJECT\\LINKAGEX\\data\\LINKAGEX_MASTER_CASES_V2.csv")
OUTPUT_DIR = Path("linkagex_audit_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. "
        "Place LINKAGEX_MASTER_CASES_V2.csv next to this notebook or update DATA_PATH."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)
print("Shape:", df.shape)
print("Unique case IDs:", df["case_id"].nunique() if "case_id" in df else "case_id missing")
print("Unique complaint texts:", df["complaint_text"].nunique() if "complaint_text" in df else "complaint_text missing")
display(df.head(3))

## 1. Schema and missingness

In [ ]:
schema = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_pct": [round(float(df[c].isna().mean() * 100), 2) for c in df.columns],
    "unique_non_null": [int(df[c].nunique(dropna=True)) for c in df.columns],
})
display(schema.sort_values(["missing_pct", "column"], ascending=[False, True]))
schema.to_csv(OUTPUT_DIR / "schema_missingness.csv", index=False)

## 2. Label distributions and rare classes

In [ ]:
TARGET_COLUMNS = [
    "event_type", "crime_category", "crime_subcategory", "attack_variant",
    "attack_pattern", "modus_operandi", "deception_method", "victim_action",
    "attacker_action", "outcome", "channel", "platform_or_service",
    "communication_direction", "payment_method", "loss_occurred"
]
TARGET_COLUMNS = [c for c in TARGET_COLUMNS if c in df.columns]

distribution_rows = []
for col in TARGET_COLUMNS:
    counts = df[col].fillna("<MISSING>").astype(str).value_counts(dropna=False)
    for label, count in counts.items():
        distribution_rows.append({
            "field": col, "label": label, "count": int(count),
            "pct": round(float(count / len(df) * 100), 3)
        })

label_distribution = pd.DataFrame(distribution_rows)
display(label_distribution.sort_values(["field", "count"], ascending=[True, True]))
label_distribution.to_csv(OUTPUT_DIR / "label_distributions.csv", index=False)

print("Lowest-support labels per field:")
for col in TARGET_COLUMNS:
    counts = df[col].fillna("<MISSING>").astype(str).value_counts()
    print(f"\n{col} (classes={len(counts)}):")
    display(counts.tail(10).rename("count").to_frame())

## 3. Annotation-source and confidence audit

In [ ]:
for col in ["annotation_source", "annotation_confidence", "nlp_status"]:
    if col in df.columns:
        print(f"\n{col}")
        display(df[col].value_counts(dropna=False).sort_index().to_frame("count"))

if "annotation_source" in df.columns:
    for target in ["crime_category", "crime_subcategory", "attack_variant", "modus_operandi",
                   "deception_method", "victim_action", "attacker_action", "outcome"]:
        if target in df.columns:
            print(f"\n{target} label diversity by annotation_source")
            display(pd.crosstab(df["annotation_source"], df[target], normalize="index").round(3))

## 4. Duplicate and near-template audit

In [ ]:
def normalize_for_fingerprint(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+", "<URL>", text)
    text = re.sub(r"\b\d{4,}\b", "<NUM>", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

if "complaint_text" in df.columns:
    df["_text_fingerprint"] = df["complaint_text"].map(normalize_for_fingerprint)
    df["_template_hash"] = df["_text_fingerprint"].map(
        lambda x: hashlib.sha1(x.encode("utf-8")).hexdigest()
    )
    print("Exact duplicate complaint texts:", int(df["complaint_text"].duplicated().sum()))
    print("Unique normalized template fingerprints:", df["_template_hash"].nunique())
    template_counts = df["_template_hash"].value_counts()
    print("Rows in normalized templates occurring more than once:",
          int(df["_template_hash"].map(template_counts).gt(1).sum()))
    repeated_hashes = template_counts[template_counts > 1].index
    repeated_templates = df[df["_template_hash"].isin(repeated_hashes)].sort_values("_template_hash")
    display(repeated_templates[["case_id", "complaint_text", "crime_category",
                                "crime_subcategory", "attack_variant", "campaign_id",
                                "incident_cluster_id", "attacker_entity_id"]].head(50))
    repeated_templates.to_csv(OUTPUT_DIR / "repeated_template_examples.csv", index=False)
    df.drop(columns=["_text_fingerprint", "_template_hash"], inplace=True)

## 5. Cross-field consistency checks (flag only; do not auto-correct)

In [ ]:
flags = pd.DataFrame(index=df.index)
flags["loss_true_amount_zero"] = (
    df.get("loss_occurred", pd.Series(False, index=df.index)).eq(True)
    & pd.to_numeric(df.get("amount_lost", pd.Series(np.nan, index=df.index)), errors="coerce").fillna(0).eq(0)
)
flags["loss_false_amount_positive"] = (
    df.get("loss_occurred", pd.Series(False, index=df.index)).eq(False)
    & pd.to_numeric(df.get("amount_lost", pd.Series(np.nan, index=df.index)), errors="coerce").fillna(0).gt(0)
)
if "event_type" in df and "outcome" in df:
    flags["attempted_but_financial_loss_outcome"] = (
        df["event_type"].astype(str).eq("ATTEMPTED_FRAUD")
        & df["outcome"].astype(str).str.contains(r"Financial loss|Unauthorized transaction", case=False, regex=True)
    )
    flags["reported_crime_but_attempt_unsuccessful"] = (
        df["event_type"].astype(str).eq("REPORTED_CRIME")
        & df["outcome"].astype(str).str.contains("Attempt unsuccessful", case=False, regex=False)
    )
if "victim_action" in df:
    flags["did_nothing_but_loss_true"] = (
        df["victim_action"].astype(str).str.contains("Did nothing", case=False, regex=False)
        & df.get("loss_occurred", pd.Series(False, index=df.index)).eq(True)
    )
if "channel" in df and "platform_or_service" in df:
    channel_norm = df["channel"].fillna("").astype(str).str.lower()
    platform_norm = df["platform_or_service"].fillna("").astype(str).str.lower()
    flags["channel_platform_different_tokens"] = (
        channel_norm.ne("") & platform_norm.ne("")
        & ~channel_norm.apply(lambda x: any(tok.strip() and tok.strip() in platform_norm.iloc[0] for tok in []))
    )
# The channel/platform flag above is intentionally not used as a definitive contradiction:
# these fields can validly represent different parts of an incident.

flag_summary = flags.sum().sort_values(ascending=False).rename("flagged_rows").to_frame()
display(flag_summary)
flagged_rows = df.loc[flags.any(axis=1)].copy()
for c in flags.columns:
    flagged_rows[c] = flags.loc[flagged_rows.index, c]
display(flagged_rows.head(100))
flagged_rows.to_csv(OUTPUT_DIR / "potential_label_inconsistencies.csv", index=False)
print("Flagged rows exported. These require human review; flags are not proof of incorrect labels.")

## 6. Linkage ground-truth audit

In [ ]:
ID_COLUMNS = [c for c in ["incident_cluster_id", "campaign_id", "attacker_entity_id"] if c in df.columns]
for col in ID_COLUMNS:
    print(f"\n{col}: missing={df[col].isna().sum()}, unique_nonmissing={df[col].nunique(dropna=True)}")
    display(df[col].value_counts(dropna=True).describe().to_frame("summary"))

available_ids = [c for c in ID_COLUMNS if c in df.columns]
linked = df.dropna(subset=available_ids) if available_ids else pd.DataFrame()
print("Rows with all linkage IDs present:", len(linked))
if len(linked):
    for a, b in [("incident_cluster_id", "campaign_id"),
                 ("campaign_id", "attacker_entity_id"),
                 ("incident_cluster_id", "attacker_entity_id")]:
        if a in linked and b in linked:
            n = linked.groupby(a)[b].nunique()
            print(f"{b} unique per {a}: min={n.min()}, median={n.median()}, max={n.max()}")
            print("One-to-one mapping fraction:", round(float((n == 1).mean()), 4))

# Do not use these IDs as NLP inputs. They are candidate labels/metadata for linkage evaluation.


## 7. Multi-label field audit

In [ ]:
# Semicolon-separated values may represent multiple labels.
for col in ["victim_action", "attacker_action", "outcome", "channel", "evidence_types"]:
    if col in df.columns:
        series = df[col].dropna().astype(str)
        multi = series.str.contains(";", regex=False)
        print(f"{col}: non-null={len(series)}, semicolon-separated={int(multi.sum())}, "
              f"pct={round(float(multi.mean()*100), 2)}%")
        if multi.any():
            display(series[multi].head(10).to_frame(col))

print("\nInterpretation rule:")
print("- Treat fields with multiple values as multi-label only after confirming delimiter/annotation semantics.")
print("- For multi-label targets, split labels consistently and report micro/macro F1.")
print("- Do not train a single-label softmax model on semicolon-combined values.")

## 8. Leakage-safe split recommendation

In [ ]:
# First choice: group by campaign/cluster/attacker ID so related incidents do not cross splits.
# Missing IDs need a stable fallback group based on normalized complaint fingerprint.
d = df.copy()
d["_fallback_group"] = d["complaint_text"].map(normalize_for_fingerprint).map(
    lambda x: hashlib.sha1(x.encode("utf-8")).hexdigest()
)

group_candidates = [c for c in ["campaign_id", "incident_cluster_id", "attacker_entity_id"] if c in d.columns]
if group_candidates:
    # Build one group key from available IDs; rows with no IDs use normalized-text fingerprint.
    def group_key(row):
        vals = [str(row[c]) for c in group_candidates if pd.notna(row[c]) and str(row[c]).strip()]
        return "ID:" + "|".join(vals) if vals else "TEXT:" + row["_fallback_group"]
    d["split_group"] = d.apply(group_key, axis=1)
else:
    d["split_group"] = "TEXT:" + d["_fallback_group"]

print("Rows:", len(d))
print("Unique split groups:", d["split_group"].nunique())
print("Largest split groups:")
display(d["split_group"].value_counts().head(15).to_frame("rows"))
print("\nUse GroupShuffleSplit or GroupKFold with split_group for evaluation.")
print("Fit any learned preprocessing only on training data.")
print("Do not feed case_id, campaign_id, incident_cluster_id, attacker_entity_id, annotation_source, or annotation_confidence as complaint-model inputs.")

## 9. Export a model-ready audit summary

In [ ]:
summary = {
    "rows": int(len(df)),
    "columns": int(df.shape[1]),
    "unique_complaint_texts": int(df["complaint_text"].nunique()) if "complaint_text" in df else None,
    "language_classes": int(df["language"].nunique()) if "language" in df else None,
    "annotation_sources": int(df["annotation_source"].nunique()) if "annotation_source" in df else None,
    "semantic_target_columns_present": [c for c in TARGET_COLUMNS if c in df.columns],
    "linkage_id_columns_present": ID_COLUMNS,
    "audit_outputs_dir": str(OUTPUT_DIR.resolve()),
}
print(summary)
pd.Series(summary, dtype="object").to_json(OUTPUT_DIR / "audit_summary.json", indent=2)
print("\nAudit complete. Review potential_label_inconsistencies.csv and repeated_template_examples.csv before training.")
